In [1]:
import pandas as pd
import numpy as np

In [2]:
name = input("Введите название инструмента(пример: AFKS, GOLD, YDEX):")

In [3]:
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/data/{name}_Williams_10years_H1.csv")


FileNotFoundError: [Errno 2] No such file or directory: '/Users/side/Desktop/Trading Chaos AI/df/data/AFKS_Williams_10years_H1.csv'

In [ ]:
df.info ()

Смотрим данные на наличие пропусков

In [ ]:
for col in df.columns:
    pct_missing = df[col].isnull().mean()
    print('{} - {}%'.format(col, round(pct_missing*100)))

в фракталах добавим бинарность, тк много пропусков

1 вход для фрактала лонг
-1 вход для фрактала шорт

0 отсутствие входа

In [ ]:
df['Fractal_Down'] = df['Fractal_Down'].notna().astype(int)
df['Fractal_Up'] = df['Fractal_Up'].notna().astype(int)

for col in df.columns:
    pct_missing = df[col].isnull().mean()
    print('{} - {}%'.format(col, round(pct_missing*100)))

In [ ]:
df. describe( include='all'). T

В классической стратегии нам нет смысла использовать объемы, поэтому убеерем их



In [ ]:
df.drop(['Volume'], axis=1, inplace=True)
df

Cделаем новый столбец "Color AO" - тоже сделаем бинарный:

- 1 зеленый - лонговый  

- -1 красный - шортовый

In [ ]:
df['Color AO'] = df['AO'].diff().apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))
df['Color AO'] = df['Color AO'].fillna(0).astype(int)

In [ ]:
print(df[['AO', 'Color AO']].head(15))
print("\nРаспределение цветов баров:")
print(df['Color AO'].value_counts().sort_index())

добавляем перемменную и признаки для Аллигатора

первый код = "что есть сигнал?"


In [ ]:
# сортируем по дате на всякий случай
df = df.sort_values("DateTime").reset_index(drop=True)


Состояние Аллигатора

In [ ]:
jaw, teeth, lips = df["Alligator_Jaw"], df["Alligator_Teeth"], df["Alligator_Lips"]

bullish = (lips > teeth) & (teeth > jaw) # бычий тренд
bearish = (jaw  > teeth) & (teeth > lips) # медвежий тренд

df["Alligator_Bullish"] = bullish.astype(int)
df["Alligator_Bearish"] = bearish.astype(int)

df["AlligatorStart_Long"]  = (bullish & ~bullish.shift(1, fill_value=False)).astype(int)
df["AlligatorStart_Short"] = (bearish & ~bearish.shift(1, fill_value=False)).astype(int)

Логика по АО

In [ ]:
df["AO_sign"] = np.where(df["Color AO"] > 0, 1, -1)

df["AO_zero_up"]   = ((df["AO"] > 0) & (df["AO"].shift(1) <= 0)).astype(int)
df["AO_zero_down"] = ((df["AO"] < 0) & (df["AO"].shift(1) >= 0)).astype(int)

# три подряд бара одного цвета
df["AO_three_green"] = (
    (df["AO_sign"]==1) &
    (df["AO_sign"].shift(1)==1) &
    (df["AO_sign"].shift(2)==1)
).astype(int)

df["AO_three_red"] = (
    (df["AO_sign"]==-1) &
    (df["AO_sign"].shift(1)==-1) &
    (df["AO_sign"].shift(2)==-1)
).astype(int)

# блюдце по AO
df["AO_saucer_up"] = (
    (df["AO"] > 0) &
    (df["AO"].shift(2) > df["AO"].shift(1)) &
    (df["AO"] > df["AO"].shift(1))
).astype(int)

df["AO_saucer_down"] = (
    (df["AO"] < 0) &
    (df["AO"].shift(2) < df["AO"].shift(1)) &
    (df["AO"] < df["AO"].shift(1))
).astype(int)

Генерация сигналов входа

In [ ]:
df["EntrySignal"] = 0
df["EntryReason"] = 0

state = "flat"
last_alligator_side = 0

цикл по всем барам:

In [ ]:
for i in range(len(df)):
    start_long  = bool(df.at[i, "AlligatorStart_Long"])
    start_short = bool(df.at[i, "AlligatorStart_Short"])

    # отслеживаем когда аллигатор открылся вверх/вниз
    if state in ("flat", "in_long", "in_short"):
        if start_long:
            state = "wait_long"; last_alligator_side = 1
        elif start_short:
            state = "wait_short"; last_alligator_side = -1

    # если появился новый «старт» аллигатора, переходим в режим ожидания сигнала AO (wait_long/wait_short)
    if state == "wait_long":
        if df.at[i, "AO_zero_up"]==1:
            df.at[i, "EntrySignal"] = 1; df.at[i, "EntryReason"] = "1"; state = "in_long" # zero line
        elif df.at[i, "AO_three_green"]==1:
            df.at[i, "EntrySignal"] = 1; df.at[i, "EntryReason"] = "2"; state = "in_long" # three colors
        elif df.at[i, "AO_saucer_up"]==1:
            df.at[i, "EntrySignal"] = 1; df.at[i, "EntryReason"] = "3"; state = "in_long" # saucer

    # В бычьем режиме ждём один из трёх паттернов AO 
    # Как только он случился – ставим EntrySignal = 1, записываем причину и считаем, что мы «в лонге»
    elif state == "wait_short":
        if df.at[i, "AO_zero_down"]==1:
            df.at[i, "EntrySignal"] = -1; df.at[i, "EntryReason"] = "1"; state = "in_short"
        elif df.at[i, "AO_three_red"]==1:
            df.at[i, "EntrySignal"] = -1; df.at[i, "EntryReason"] = "2"; state = "in_short"
        elif df.at[i, "AO_saucer_down"]==1:
            df.at[i, "EntrySignal"] = -1; df.at[i, "EntryReason"] = "3"; state = "in_short"
    # То же для шорта
    if state in ("in_long", "wait_long") and start_short:
        state = "wait_short"; last_alligator_side = -1
        # Если во время ожидания/позиции появился противоположный аллигатор 
        # переключаемся на ожидание сигнала в новую сторону.
    if state in ("in_short", "wait_short") and start_long:
        state = "wait_long"; last_alligator_side = 1

Подтверждение фракталов

Фрактал у Вильямса подтверждается спустя 2 бара. Поэтому здесь берётся значение фрактала 2 бара назад и переносится вперёд – чтобы на текущем баре понимать, что фрактал уже подтверждён.

In [ ]:
df["Fractal_Up_conf"]   = df["Fractal_Up"].shift(2).fillna(0).astype(int)
df["Fractal_Down_conf"] = df["Fractal_Down"].shift(2).fillna(0).astype(int)

Логика добавления к позиции (AddOn)

In [ ]:
df["AddOn_Anchor_Level"] = np.nan
df["AddOn_Anchor_IsUp"]  = np.nan
df["AddOn_Size_Pct"]     = np.nan
waiting_anchor = False
pos_side = 0

# Колонки для «якоря» добавочных входов: 
# уровень цены, направление (1 – вверх, 0 – вниз), размер добавки (в процентах).
for i in range(len(df)):
    sig = int(df.at[i, "EntrySignal"])
    if sig != 0:
        pos_side = sig
        waiting_anchor = True
        continue
    # когда появился вход (EntrySignal ≠ 0) — запоминаем направление позиции и включаем режим waiting_anchor:
    #  ждём подходящего фрактала для постановки уровня добавки
    if waiting_anchor and pos_side == 1 and df.at[i, "Fractal_Up_conf"] == 1:
        df.at[i, "AddOn_Anchor_Level"] = df.at[i, "High"]
        df.at[i, "AddOn_Anchor_IsUp"]  = 1
        df.at[i, "AddOn_Size_Pct"]     = 0.30
        waiting_anchor = False
    # Если в лонге и появился подтверждённый верхний фрактал – его максимум становится AddOn_Anchor_Level,
    #  направление вверх, размер 30% от базовой позиции.
    if waiting_anchor and pos_side == -1 and df.at[i, "Fractal_Down_conf"] == 1:
        df.at[i, "AddOn_Anchor_Level"] = df.at[i, "Low"]
        df.at[i, "AddOn_Anchor_IsUp"]  = 0
        df.at[i, "AddOn_Size_Pct"]     = 0.30
        waiting_anchor = False

Аналогично для шорта (минимум фрактала).

In [ ]:
df["AddOn_Anchor_Level"] = df["AddOn_Anchor_Level"].ffill()
df["AddOn_Anchor_IsUp"]  = df["AddOn_Anchor_IsUp"].ffill()
df["AddOn_Size_Pct"]     = df["AddOn_Size_Pct"].ffill()


In [ ]:
df

второй код = «из чего модель будет его угадывать?».


In [ ]:
df = df.copy().sort_values("DateTime").reset_index(drop=True)

# 1) Подтверждённые фракталы (каузально)
df["Fractal_Up_conf"]   = df["Fractal_Up"].shift(2).fillna(0).astype(int)
df["Fractal_Down_conf"] = df["Fractal_Down"].shift(2).fillna(0).astype(int)

# 2) Якорь добора — первый фрактал в сторону позиции ПОСЛЕ входа
df["AddOn_Anchor_Level"] = np.nan
df["AddOn_Anchor_IsUp"]  = np.nan  # 1=верхний, 0=нижний
df["AddOn_Size_Pct"]     = np.nan
df["AddOn_Ready"]        = 0       # «якорь найден, добор ещё не выполнен»
df["AddOn_Triggered"]    = 0       # «добор исполнен по цене якоря»

in_pos = 0           # +1 long, -1 short, 0 flat
anchor_set = False   # найден ли якорь добора для текущей сделки
addon_done = False   # добор уже выполнен?

for i in range(len(df)):
    sig = int(df.at[i, "EntrySignal"]) if "EntrySignal" in df.columns else 0

    # вход открывает новую «сделку»
    if in_pos == 0 and sig != 0:
        in_pos = sig
        anchor_set = False
        addon_done = False
        continue  # на самом баре входа якорь ещё не может появиться (фракталы подтверждаются позже)

    # если в позиции и якорь ещё не выбран — ловим ПЕРВЫЙ подтверждённый фрактал в сторону сделки
    if in_pos == 1 and not anchor_set and df.at[i, "Fractal_Up_conf"] == 1:
        df.at[i, "AddOn_Anchor_Level"] = df.at[i, "High"]
        df.at[i, "AddOn_Anchor_IsUp"]  = 1
        df.at[i, "AddOn_Size_Pct"]     = 0.30
        df.at[i, "AddOn_Ready"]        = 1
        anchor_set = True
    elif in_pos == -1 and not anchor_set and df.at[i, "Fractal_Down_conf"] == 1:
        df.at[i, "AddOn_Anchor_Level"] = df.at[i, "Low"]
        df.at[i, "AddOn_Anchor_IsUp"]  = 0
        df.at[i, "AddOn_Size_Pct"]     = 0.30
        df.at[i, "AddOn_Ready"]        = 1
        anchor_set = True

    # если якорь выбран, проверим срабатывание (по High/Low)
    if anchor_set and not addon_done:
        anchor = df.at[i, "AddOn_Anchor_Level"]
        if in_pos == 1 and not pd.isna(anchor) and df.at[i, "High"] >= anchor:
            df.at[i, "AddOn_Triggered"] = 1
            addon_done = True
        elif in_pos == -1 and not pd.isna(anchor) and df.at[i, "Low"] <= anchor:
            df.at[i, "AddOn_Triggered"] = 1
            addon_done = True

    # выход по развороту аллигатора обнуляет состояние (если у тебя уже есть ExitSignal — можно использовать его)
    # Здесь считаем flip как смену устойчивого порядка:
    jaw, teeth, lips = df["Alligator_Jaw"], df["Alligator_Teeth"], df["Alligator_Lips"]
    bullish = (lips > teeth) & (teeth > jaw)
    bearish = (jaw  > teeth) & (teeth > lips)
    start_long  = bool(bullish.iloc[i] and not bullish.shift(1, fill_value=False).iloc[i])
    start_short = bool(bearish.iloc[i] and not bearish.shift(1, fill_value=False).iloc[i])

    if in_pos == 1 and start_short:
        in_pos = 0; anchor_set = False; addon_done = False
    elif in_pos == -1 and start_long:
        in_pos = 0; anchor_set = False; addon_done = False

# протащим постоянные значения якоря вперёд до конца сделки (удобно для исполнителя)
df["AddOn_Anchor_Level"] = df["AddOn_Anchor_Level"].ffill()
df["AddOn_Anchor_IsUp"]  = df["AddOn_Anchor_IsUp"].ffill()
df["AddOn_Size_Pct"]     = df["AddOn_Size_Pct"].ffill()


In [ ]:
df

Предобработка временных меток в DataFrame

In [ ]:
df = df.copy().sort_values("DateTime").reset_index(drop=True)
df["DateTime"] = pd.to_datetime(df["DateTime"], errors="coerce")
df = df[df["DateTime"].notna()].reset_index(drop=True)


In [ ]:
df

In [ ]:
df.info()

In [ ]:
df['EntryReason'] = df['EntryReason'].astype('int32')

In [ ]:
for col in df.columns:
    pct_missing = df[col].isnull().mean()
    print('{} - {}%'.format(col, round(pct_missing*100)))

In [ ]:
df.describe(include='all').T

In [ ]:
df['EntrySignal'].value_counts()

In [ ]:
df['EntryReason'].value_counts()

In [ ]:
df.to_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv", index=False)